In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

In [2]:
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
)

In [3]:
import plotly.graph_objects as go

In [4]:
from statsmodels.tsa.arima.model import ARIMA

In [5]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [6]:
import numpy as np


In [79]:
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
import joblib

In [8]:
df = pd.read_csv(
    "../data/raw/amsterdam_2023-01-01_2024-12-31.csv"
)

In [9]:
df.head()

,time,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,city
0,2023-01-01,12.1,14.8,10.6,4.5,43.5,Amsterdam
1,2023-01-02,8.8,11.2,4.3,5.1,22.2,Amsterdam
2,2023-01-03,5.1,8.4,1.4,1.1,28.6,Amsterdam
3,2023-01-04,10.7,12.3,8.3,12.1,40.0,Amsterdam
4,2023-01-05,9.9,10.9,9.2,2.1,30.3,Amsterdam


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   time                 731 non-null    object 
 1   temperature_2m_mean  731 non-null    float64
 2   temperature_2m_max   731 non-null    float64
 3   temperature_2m_min   731 non-null    float64
 4   precipitation_sum    731 non-null    float64
 5   wind_speed_10m_max   731 non-null    float64
 6   city                 731 non-null    object 
dtypes: float64(5), object(2)
memory usage: 40.1+ KB


In [11]:
df["time"] = pd.to_datetime(df["time"])

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   time                 731 non-null    datetime64[ns]
 1   temperature_2m_mean  731 non-null    float64       
 2   temperature_2m_max   731 non-null    float64       
 3   temperature_2m_min   731 non-null    float64       
 4   precipitation_sum    731 non-null    float64       
 5   wind_speed_10m_max   731 non-null    float64       
 6   city                 731 non-null    object        
dtypes: datetime64[ns](1), float64(5), object(1)
memory usage: 40.1+ KB


In [13]:
df.describe()

,time,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max
count,731,731.000000,731.000000,731.000000,731.000000,731.000000
mean,2024-01-01 00:00:00,11.535705,14.693844,8.475239,3.277018,22.415458
min,2023-01-01 00:00:00,-3.000000,-1.100000,-6.000000,0.000000,7.400000
25%,2023-07-02 12:00:00,7.500000,10.050000,4.800000,0.000000,16.200000
50%,2024-01-01 00:00:00,11.000000,14.100000,8.600000,1.100000,21.500000
75%,2024-07-01 12:00:00,16.200000,19.500000,12.600000,4.350000,27.000000
max,2024-12-31 00:00:00,23.900000,31.200000,20.000000,37.900000,60.900000
std,NaN,5.609358,6.313393,5.258741,5.118719,8.345856


# 1. lag feature

Lag = прошлое значение временного ряда.

In [14]:
df["lag_1"] = df["temperature_2m_mean"].shift(1)

df["lag_2"] = df["temperature_2m_mean"].shift(2)

df["lag_7"] = df["temperature_2m_mean"].shift(7)

In [15]:
df.head(10)

,time,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,city,lag_1,lag_2,lag_7
0,2023-01-01,12.1,14.8,10.6,4.5,43.5,Amsterdam,NaN,NaN,NaN
1,2023-01-02,8.8,11.2,4.3,5.1,22.2,Amsterdam,12.1,NaN,NaN
2,2023-01-03,5.1,8.4,1.4,1.1,28.6,Amsterdam,8.8,12.1,NaN
3,2023-01-04,10.7,12.3,8.3,12.1,40.0,Amsterdam,5.1,8.8,NaN
4,2023-01-05,9.9,10.9,9.2,2.1,30.3,Amsterdam,10.7,5.1,NaN
5,2023-01-06,9.7,10.6,8.4,1.7,32.1,Amsterdam,9.9,10.7,NaN
6,2023-01-07,10.0,11.2,8.3,0.8,29.0,Amsterdam,9.7,9.9,NaN
7,2023-01-08,8.2,10.1,6.7,0.4,27.0,Amsterdam,10.0,9.7,12.1
8,2023-01-09,6.6,8.4,4.6,4.9,23.9,Amsterdam,8.2,10.0,8.8
9,2023-01-10,6.6,10.3,4.9,7.2,32.6,Amsterdam,6.6,8.2,5.1


# 2. Rolling Mean
Что такое rolling window

Это:

среднее за последние N дней


In [16]:
df["rolling_mean_7"] = (
    df["temperature_2m_mean"]
    .rolling(window=7)
    .mean()
)

In [17]:
df.head(10)

,time,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,city,lag_1,lag_2,lag_7,rolling_mean_7
0,2023-01-01,12.1,14.8,10.6,4.5,43.5,Amsterdam,NaN,NaN,NaN,NaN
1,2023-01-02,8.8,11.2,4.3,5.1,22.2,Amsterdam,12.1,NaN,NaN,NaN
2,2023-01-03,5.1,8.4,1.4,1.1,28.6,Amsterdam,8.8,12.1,NaN,NaN
3,2023-01-04,10.7,12.3,8.3,12.1,40.0,Amsterdam,5.1,8.8,NaN,NaN
4,2023-01-05,9.9,10.9,9.2,2.1,30.3,Amsterdam,10.7,5.1,NaN,NaN
5,2023-01-06,9.7,10.6,8.4,1.7,32.1,Amsterdam,9.9,10.7,NaN,NaN
6,2023-01-07,10.0,11.2,8.3,0.8,29.0,Amsterdam,9.7,9.9,NaN,9.471429
7,2023-01-08,8.2,10.1,6.7,0.4,27.0,Amsterdam,10.0,9.7,12.1,8.914286
8,2023-01-09,6.6,8.4,4.6,4.9,23.9,Amsterdam,8.2,10.0,8.8,8.600000
9,2023-01-10,6.6,10.3,4.9,7.2,32.6,Amsterdam,6.6,8.2,5.1,8.814286


In [18]:
df = df.dropna()

In [19]:
df.shape


(724, 11)

# 3. ML

In [20]:
feature_columns = [
    "lag_1",
    "lag_2",
    "lag_7",
    "rolling_mean_7",
    "precipitation_sum",
    "wind_speed_10m_max",
]

X = df[feature_columns]

y = df["temperature_2m_mean"]

In [21]:
X.head()

,lag_1,lag_2,lag_7,rolling_mean_7,precipitation_sum,wind_speed_10m_max
7,10.0,9.7,12.1,8.914286,0.4,27.0
8,8.2,10.0,8.8,8.600000,4.9,23.9
9,6.6,8.2,5.1,8.814286,7.2,32.6
10,6.6,6.6,10.7,8.657143,5.6,37.3
11,9.6,6.6,9.9,8.671429,25.8,39.7


In [22]:
y.head()

7      8.2
8      6.6
9      6.6
10     9.6
11    10.0
Name: temperature_2m_mean, dtype: float64

In [23]:
# разбивка по времени
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index] # старые индексы для обучения
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

In [24]:
X_train.shape, X_test.shape

((579, 6), (145, 6))

In [25]:
y_train.shape, y_test.shape

((579,), (145,))

# Model 1 - Random Forest

In [26]:
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
)

In [27]:
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [28]:
predictions = model.predict(X_test)

In [29]:
mae_rf = mean_absolute_error(y_test, predictions)

rmse_rf = root_mean_squared_error(y_test, predictions)

print("MAE_rf:", mae_rf)
print("RMSE_rf:", rmse_rf)

MAE_rf: 1.6246344827586203
RMSE_rf: 2.1140881396169284


В среднем модель ошибается на 1.6 С
RMSE выше. То есть иногда модель ошибается силнее обычного 

In [30]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y=y_test.values,
        mode="lines",
        name="Actual",
    )
)

fig.add_trace(
    go.Scatter(
        y=predictions,
        mode="lines",
        name="Predicted",
    )
)

fig.update_layout(
    title="Weather Forecast: Actual vs Predicted",
    xaxis_title="Time",
    yaxis_title="Temperature",
    template="plotly_white",
)

fig.show()

# Model 2 ARIMA

In [31]:
series = (
    df
    .set_index("time")["temperature_2m_mean"]
    .asfreq("D")
)

In [32]:
train_size = int(len(series) * 0.8)

train = series.iloc[:train_size]
test = series.iloc[train_size:]

In [33]:
arima_model = ARIMA(
    train,
    order=(7, 1, 2),
)

In [34]:
arima_fitted = arima_model.fit()

In [35]:
arima_predictions = arima_fitted.forecast(
    steps=len(test)
)

In [36]:
mae_arima = mean_absolute_error(
    test,
    arima_predictions,
)

rmse_arima = root_mean_squared_error(
    test,
    arima_predictions,
)

print("MAE_ARIMA:", mae_arima)
print("RMSE_ARIMA:", rmse_arima)

MAE_ARIMA: 7.576433174292028
RMSE_ARIMA: 8.985002042396736


# Сравнение результатов

In [37]:
results_df = pd.DataFrame({
    "Model": ["RandomForest", "ARIMA"],
    "MAE": [mae_rf, mae_arima],
    "RMSE": [rmse_rf, rmse_arima],
})

results_df

,Model,MAE,RMSE
0,RandomForest,1.624634,2.114088
1,ARIMA,7.576433,8.985002


RandomForest сильно лучше. Причём разница огромная.

Почему ARIMA проиграла

Это очень важное понимание для forecasting.

Причина 1

ARIMA использует только: сам временной ряд
А RandomForest использовал дополнительные признаки
lag_1
lag_2
lag_7
rolling_mean_7
precipitation
wind
То есть RF видел больше информации

Причина 2
Weather — сложный нелинейный процесс
ARIMA хорошо работает, когда:
ряд относительно линейный и стационарный
Но погода:
❌ нелинейна
❌ шумная
❌ имеет complex seasonality
❌ имеет резкие скачки

Tree ensembles часто лучше для такого
Потому что они умеют:
✅ nonlinear patterns
✅ feature interactions
✅ thresholds
✅ complex dependencies

Причина 3
Мы использовали обычную ARIMA
А weather имеет:seasonality
Поэтому SARIMA вероятно была бы лучше

In [38]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        y=test.values,
        mode="lines",
        name="Actual",
    )
)

fig.add_trace(
    go.Scatter(
        y=predictions,
        mode="lines",
        name="RandomForest",
    )
)

fig.add_trace(
    go.Scatter(
        y=arima_predictions,
        mode="lines",
        name="ARIMA",
    )
)

fig.update_layout(
    title="Forecast Comparison",
    xaxis_title="Time",
    yaxis_title="Temperature",
    template="plotly_white",
)

fig.show()

Обычная ARIMA показала низкое качество прогноза и склонность к усреднению ряда, поскольку модель не учитывает сезонные компоненты и плохо описывает сложную нелинейную динамику погодных данных.

# Model 3 SARIMA

In [39]:
sarima_model = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7), #недельная сезонность
)

In [40]:
sarima_fitted = sarima_model.fit()

In [41]:
sarima_predictions = sarima_fitted.forecast(
    steps=len(test)
)

In [42]:
mae_sarima = mean_absolute_error(
    test,
    sarima_predictions,
)

rmse_sarima = root_mean_squared_error(
    test,
    sarima_predictions,
)

print("MAE_SARIMA:", mae_sarima)
print("RMSE_SARIMA:", rmse_sarima)

MAE_SARIMA: 10.954473670652758
RMSE_SARIMA: 12.846872176387288


# Сравнение результатов

In [43]:
results_df = pd.DataFrame({
    "Model": [
        "RandomForest",
        "ARIMA",
        "SARIMA",
    ],
    "MAE": [
        mae_rf,
        mae_arima,
        mae_sarima,
    ],
    "RMSE": [
        rmse_rf,
        rmse_arima,
        rmse_sarima,
    ],
})

results_df = results_df.round(2)

results_df

,Model,MAE,RMSE
0,RandomForest,1.62,2.11
1,ARIMA,7.58,8.99
2,SARIMA,10.95,12.85


Несмотря на популярность SARIMA для временных рядов, она плохо подходит для сложных нелинейных погодных данных с высокой вариативностью.

# Model 4 LSTM

In [60]:
# Преобразует pandas Series → numpy array. 
# PyTorch работает с: numpy / tensors
# а не с pandas напрямую.
temperature_series = (
    df["temperature_2m_mean"]
    .values
)

In [61]:
scaler = MinMaxScaler()

scaled_series = scaler.fit_transform(
    temperature_series.reshape(-1, 1)
)

In [62]:
# Используем: последнюю неделю для прогноза следующего дня.
sequence_length = 7

In [63]:
X_lstm = []
y_lstm = []

for i in range(sequence_length, len(scaled_series)):
    
    X_lstm.append(
        scaled_series[i-sequence_length:i]
    )
    
    y_lstm.append(
        scaled_series[i]
    )

X_lstm = np.array(X_lstm)
y_lstm = np.array(y_lstm)

In [64]:
X_lstm.shape, y_lstm.shape

((717, 7, 1), (717, 1))

In [65]:
# PyTorch не работает напрямую с numpy. переводим в tensors
X_lstm = torch.tensor(
    X_lstm,
    dtype=torch.float32,
)

y_lstm = torch.tensor(
    y_lstm,
    dtype=torch.float32,
)

In [66]:
split_index = int(len(X_lstm) * 0.8)

X_train_lstm = X_lstm[:split_index]
X_test_lstm = X_lstm[split_index:]

y_train_lstm = y_lstm[:split_index]
y_test_lstm = y_lstm[split_index:]

In [67]:
X_train_lstm.shape, X_test_lstm.shape

(torch.Size([573, 7, 1]), torch.Size([144, 7, 1]))

In [68]:
class LSTMModel(nn.Module):
    
    def __init__(self):
        
        super().__init__()
        
        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=32,
            batch_first=True,
        )
        
        self.fc = nn.Linear(
            32,
            1,
        )

    def forward(self, x):
        
        output, (hidden, cell) = self.lstm(x)
        
        hidden = hidden[-1]
        
        prediction = self.fc(hidden)
        
        return prediction

In [69]:
lstm_model = LSTMModel()

In [70]:
lstm_model

LSTMModel(
  (lstm): LSTM(1, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)

In [71]:
criterion = nn.MSELoss()

In [72]:
optimizer = torch.optim.Adam(
    lstm_model.parameters(),
    lr=0.001,
)

In [73]:
epochs = 100

for epoch in range(epochs):
    
    predictions = lstm_model(X_train_lstm)
    
    predictions = predictions.squeeze()
    
    loss = criterion(
        predictions,
        y_train_lstm,
    )
    
    optimizer.zero_grad()
    
    loss.backward()
    
    optimizer.step()
    
    if epoch % 10 == 0:
        print(
            f"Epoch {epoch}, Loss: {loss.item():.4f}"
        )

/Users/dinaragali/2_ds_mentor/0_projects/1-weather-forecast-service/.venv/lib/python3.10/site-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([573, 1])) that is different to the input size (torch.Size([573])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 0, Loss: 0.1911
Epoch 10, Loss: 0.1118
Epoch 20, Loss: 0.0616
Epoch 30, Loss: 0.0457
Epoch 40, Loss: 0.0480
Epoch 50, Loss: 0.0456
Epoch 60, Loss: 0.0458
Epoch 70, Loss: 0.0455
Epoch 80, Loss: 0.0455
Epoch 90, Loss: 0.0454


In [74]:
lstm_model.eval()

LSTMModel(
  (lstm): LSTM(1, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)

In [75]:
with torch.no_grad():
    
    lstm_predictions = lstm_model(
        X_test_lstm
    )

In [76]:
lstm_predictions = (
    lstm_predictions
    .squeeze()
    .numpy()
)

y_test_real = scaler.inverse_transform(
    y_test_lstm.reshape(-1, 1)
).flatten()

lstm_predictions_real = scaler.inverse_transform(
    lstm_predictions.reshape(-1, 1)
).flatten()

In [77]:
mae_lstm = mean_absolute_error(
    y_test_real,
    lstm_predictions_real,
)

rmse_lstm = root_mean_squared_error(
    y_test_real,
    lstm_predictions_real,
)

print("MAE_LSTM:", mae_lstm)
print("RMSE_LSTM:", rmse_lstm)

MAE_LSTM: 4.229414325197123
RMSE_LSTM: 4.981986307487042


# Сравнение 

In [78]:
results_df = pd.DataFrame({
    "Model": [
        "RandomForest",
        "ARIMA",
        "SARIMA",
        "LSTM",
    ],
    "MAE": [
        mae_rf,
        mae_arima,
        mae_sarima,
        mae_lstm,
    ],
    "RMSE": [
        rmse_rf,
        rmse_arima,
        rmse_sarima,
        rmse_lstm,
    ],
})

results_df = results_df.round(2)

results_df

,Model,MAE,RMSE
0,RandomForest,1.62,2.11
1,ARIMA,7.58,8.99
2,SARIMA,10.95,12.85
3,LSTM,4.23,4.98


# Сохраняем модели

In [80]:
joblib.dump(
    model,
    "../backend/app/artifacts/random_forest.pkl",
)

['../backend/app/artifacts/random_forest.pkl']